[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/NoPoSplat_Colab.ipynb)  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/NoPoSplat_Colab.ipynb)


# 🖼️ NoPoSplat — 2-3 Photos → 3D Gaussian Splat (MIT, ICLR 2025)

**NoPoSplat** is a feed-forward model that takes 2-3 unposed, uncalibrated images of a scene
and produces a complete 3D Gaussian Splat scene in a **single forward pass** — no COLMAP,
no bundle adjustment, no per-scene optimization. This is the closest open-source equivalent
to a Luma-style "upload photos, get 3DGS" workflow, and it runs in **~10 seconds on a T4**.

* **Paper:** [arXiv 2410.24207](https://arxiv.org/abs/2410.24207) &nbsp;·&nbsp;
  **Project page:** [noposplat.github.io](https://noposplat.github.io/) &nbsp;·&nbsp;
  **Code:** [github.com/cvg/NoPoSplat](https://github.com/cvg/NoPoSplat) &nbsp;·&nbsp;
  **Checkpoints:** [🤗&nbsp;botaoye/NoPoSplat](https://huggingface.co/botaoye/NoPoSplat)
* **Authors:** Botao Ye, Sifei Liu, Haofei Xu, Xueting Li, Marc Pollefeys, Ming-Hsuan Yang, Songyou Peng (NVIDIA + ETH Zurich)
* **Conference:** ICLR 2025 (Oral)
* **License:** This notebook is **MIT-licensed**. The upstream
  [cvg/NoPoSplat](https://github.com/cvg/NoPoSplat) code is MIT. The MASt3R backbone weights
  ([🤗&nbsp;naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric](https://huggingface.co/naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric))
  are **CC BY-NC-SA 4.0** — we only consume them for inference (no training, no
  redistribution), and the 3DGS scene you produce is yours. See "License" below.

## How it works

The model is a ViT-L encoder (built on the MASt3R backbone) with **4 heads**:
1. A **3D point head** (DPT) that predicts per-pixel world-space points
2. A **Gaussian parameter head** (DPT-GS) that predicts per-pixel opacity + scale + rotation + SH coefficients
3. A **second view head** that processes the second image in the pair
4. A **Unified Gaussian Adapter** that fuses both views' predictions into a single 3DGS scene in a canonical frame

Critically, NoPoSplat is **pose-free**: it does not require camera extrinsics or intrinsics
as input. The "canonical" coordinate system is determined by the model itself, and the
predicted per-Gaussian poses can be extracted as a byproduct. The encoder runs in **one
forward pass** — no test-time optimization, no iterative refinement.

## What you get

* **Input:** 2 or 3 images (`.jpg` / `.png`) of the same scene from different viewpoints
* **Output:** a real 3DGS scene (`.ply`, ~200-500 MB for 30k-100k Gaussians) + predicted camera poses
* **Runtime:** ~10 seconds for 2 views at 256×256, ~20-30 seconds for 2 views at 512×512 (T4)
* **VRAM:** ~6-10 GB on T4 (model is ~1.1 B parameters at bf16)

## Where this fits in our 3DGS pipeline

```
NoPoSPlat (this notebook)        TripoSplat (image → 3DGS, MIT, single image)
        │                                  │
        ▼                                  ▼
        └── .ply  ──┐         ┌── .ply ────┘
                    ▼         ▼
            ┌───── SplatTransform_Colab (SOG/SPLAT/SPZ/GLB compression) ─────┐
            │              + Asset_Library_Browser_Colab (browse/ship)       │
            └─────────────────────────────────────────────────────────────────┘
                                       │
                                       ▼
                            Three.js / WebGPU game engine
```

For **single-image** 3DGS, use [TripoSplat](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/TripoSplat_Colab.ipynb)
(MIT, VAST-AI). For **video → 3DGS** with proper per-scene optimization, see
[WildGaussianSplatting_Colab](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/WildGaussianSplatting_Colab.ipynb)
(non-commercial, INRIA, 5-15 min runtime).

## License

* **Notebook + code:** MIT (upstream [cvg/NoPoSplat](https://github.com/cvg/NoPoSplat))
* **MASt3R backbone weights (CC BY-NC-SA 4.0):** the ViT-L weights that NoPoSplat's
  encoder was fine-tuned from. Used for inference only. If you need a fully-commercial
  pipeline, the [WildGaussianSplatting](../WildGaussianSplatting_Colab.ipynb) notebook uses
  the same backbone but a different downstream path — both inherit this constraint.
* **Your output (.ply, camera poses):** yours. No restriction.

## Requirements

* **GPU:** NVIDIA, ≥ 6 GB VRAM (T4 15 GB works comfortably)
* **RAM:** ≥ 12 GB
* **Disk:** ≈ 8 GB free (PyTorch + CUDA + 2.3 GB checkpoint + working space)
* **Time on first run:** 5-8 min (PyTorch + diff-gaussian-rasterization compile + RoPE kernel + checkpoint download)
* **Time on subsequent runs:** 1-2 min (everything cached in your Drive)


In [ ]:
#@title STEP 1 — Install dependencies, clone repo, download checkpoint
"""
• Installs torch 2.4.1 + cu121, torchvision 0.19.1 (pinned — newer breaks
  torchvision.transforms.functional imports that NoPoSplat uses)
• Clones cvg/NoPoSplat (MIT) into Drive cache (submodules not needed for inference)
• Installs all requirements: diff-gaussian-rasterization-w-pose, einops, jaxtyping,
  e3nn, lpips, plyfile, etc.
• Compiles RoPE CUDA kernels (1-2 min, optional but recommended for speed)
• Downloads the `mixRe10kDl3dv_512x512.ckpt` checkpoint (2.3 GB) to Drive cache
"""
import os, sys, time, json, subprocess, shutil, pathlib, traceback
from tqdm.auto import tqdm

print('='*72)
print('NoPoSplat — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda or "none"})')
    print(f'  CUDA avail   : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}        : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected — inference will be very slow on CPU')
except ImportError:
    torch = None
    print('  torch not yet installed (will be installed below)')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    drive_root = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/NoPoSplat')
    drive_root.mkdir(parents=True, exist_ok=True)
    print(f'  Drive cache  : {drive_root}')
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(drive_root / 'huggingface')
else:
    drive_root = pathlib.Path('/content/_noposplat_cache')
    drive_root.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(drive_root / 'huggingface')
    print(f'  Local cache  : {drive_root}  (lost on runtime disconnect)')

REPO_DIR    = drive_root / 'NoPoSplat'
CKPT_DIR    = drive_root / 'checkpoints'
WEIGHTS_DIR = drive_root / 'pretrained_weights'
t_total = time.time()

# 1. PyTorch (cu121, torch 2.4.1, torchvision 0.19.1) ─────────────────
if torch is None or not torch.cuda.is_available() or not torch.__version__.startswith('2.4'):
    print('\n[1/5] Installing PyTorch 2.4.1 + cu121 ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
         'torch==2.4.1', 'torchvision==0.19.1', 'torchaudio==2.4.1',
         '--index-url', 'https://download.pytorch.org/whl/cu121'],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-1500:])
        raise RuntimeError('PyTorch install failed')
    print(f'  PyTorch installed in {time.time()-t0:.0f}s')

# 2. Clone the NoPoSplat repo ────────────────────────────────────────
if not (REPO_DIR / '.git').exists():
    print(f'\n[2/5] Cloning cvg/NoPoSplat into {REPO_DIR} ...')
    t0 = time.time()
    r = subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/cvg/NoPoSplat.git',
         str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('  git clone failed:', r.stderr[-500:])
        raise RuntimeError('git clone failed')
    print(f'  cloned in {time.time()-t0:.0f}s')
else:
    print(f'\n[2/5] Reusing cached repo at {REPO_DIR}')

sys.path.insert(0, str(REPO_DIR / 'src'))

# 3. Install requirements (skip the broken entries: `sql` is a typo) ──
print('\n[3/5] Installing NoPoSplat requirements ...')
t0 = time.time()
req = (REPO_DIR / 'requirements.txt').read_text()
req = chr(10).join(l for l in req.split(chr(10)) if l.strip() and not l.strip().startswith('#') and l.strip() != 'sql')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-build-isolation'] +
    req.split(chr(10)),
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
print(f'  requirements installed in {time.time()-t0:.0f}s')

# 4. Compile RoPE CUDA kernel (optional, ~1-2 min) ─────────────────
rope_dir = REPO_DIR / 'src' / 'model' / 'encoder' / 'backbone' / 'croco' / 'curope'
rope_so = rope_dir / 'curope' / 'curope.so'
rope_dir.mkdir(parents=True, exist_ok=True)
if rope_so.exists() or (rope_dir / 'curope' / '__init__.py').exists() and \
   any(rope_dir.rglob('*.so')):
    print('\n[4/5] Reusing cached RoPE kernel build')
else:
    print('\n[4/5] Compiling RoPE CUDA kernel (1-2 min) ...')
    t0 = time.time()
    try:
        # src/model/encoder/backbone/croco/curope/ — that's the actual path
        actual = REPO_DIR / 'src' / 'model' / 'encoder' / 'backbone' / 'croco' / 'curope'
        if not actual.exists():
            print('  curope path not found — skipping (encoder will use Python fallback)')
        else:
            subprocess.check_call(
                [sys.executable, 'setup.py', 'build_ext', '--inplace'],
                cwd=str(actual), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
            )
            print(f'  RoPE kernel built in {time.time()-t0:.0f}s')
    except Exception as e:
        print(f'  RoPE kernel build failed (non-fatal): {e}')

# 5. Download the NoPoSplat checkpoint ──────────────────────────────
print('\n[5/5] Downloading NoPoSplat checkpoint ...')
from huggingface_hub import hf_hub_download
t0 = time.time()
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = WEIGHTS_DIR / 'mixRe10kDl3dv_512x512.ckpt'
if ckpt_path.exists() and ckpt_path.stat().st_size > 2_000_000_000:
    print(f'  cached : mixRe10kDl3dv_512x512.ckpt  ({ckpt_path.stat().st_size//(1024*1024)} MB)')
else:
    print('  download : mixRe10kDl3dv_512x512.ckpt  (≈2.3 GB, 1-3 min)')
    print('  downloading with progress bar (1-3 min for 2.3 GB)')
    # huggingface_hub's hf_hub_download shows a tqdm progress bar by default
    p = hf_hub_download(
        repo_id='botaoye/NoPoSplat',
        filename='mixRe10kDl3dv_512x512.ckpt',
        local_dir=str(WEIGHTS_DIR),
    )
print(f'  checkpoint ready in {time.time()-t0:.0f}s')

print()
print('='*72)
print(f'  STEP 1 complete  (total {time.time()-t_total:.0f}s)')
print('  Next: run STEP 2 (imports + lazy encoder load)')
print('='*72)


In [ ]:
#@title STEP 2 — Imports & lazy encoder load
"""
Loads the upstream `src.*` modules (one-time cost). Builds the encoder
with `get_encoder(cfg)` and loads the checkpoint via `checkpoint_filter_fn`.
Defines `infer(images, ckpt_path) -> (gaussians, poses)` which the UI and
batch cells both call.
"""
import sys, os, time, json, pathlib, warnings
warnings.filterwarnings('ignore')

REPO_DIR = pathlib.Path(os.environ.get('AEI_NP_REPO',
                    str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/NoPoSplat/NoPoSplat'))))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))
print(f'  repo        : {REPO_DIR}')
print(f'  src on path : {(REPO_DIR / "src").exists()}')
print()

import torch
import numpy as np
from omegaconf import OmegaConf
print('  Loading src.* modules (one-time cost) ...')
t0 = time.time()
try:
    from src.model.encoder import get_encoder
    print(f'  src.model.encoder   ✓')
except Exception as e:
    print(f'  src.model.encoder   ✗  {type(e).__name__}: {e}')
    raise
from src.misc.weight_modify import checkpoint_filter_fn
print(f'  src.misc.weight_modify ✓')
from src.dataset.shims.normalize_shim import apply_normalize_shim
print(f'  src.dataset.shims.normalize_shim ✓')
from src.model.encoder.common.gaussian_adapter import Gaussians
print(f'  src.model.encoder.common.gaussian_adapter ✓')
print(f'  total import: {time.time()-t0:.1f}s')
print()

# Lazy encoder cache — avoids re-loading on every Gradio click
_ENCODER_CACHE = {}
def get_cached_encoder(ckpt_path, image_size=512, max_views=3):
    """Load NoPoSplat encoder + checkpoint. Cached per (ckpt, size, max_views)."""
    key = (str(ckpt_path), image_size, max_views)
    if key in _ENCODER_CACHE:
        return _ENCODER_CACHE[key]
    print(f'  Loading NoPoSplat encoder from {ckpt_path} ...')
    t0 = time.time()
    # Load the right config based on the checkpoint name
    ckpt_name = pathlib.Path(ckpt_path).stem
    cfg_yaml = REPO_DIR / 'config' / 'experiment' / (
        're10k_dl3dv_512x512.yaml' if '512' in ckpt_name else
        're10k_3view.yaml' if '3view' in ckpt_name or '3views' in ckpt_name else
        're10k_dl3dv.yaml'
    )
    cfg = OmegaConf.load(cfg_yaml)
    encoder, _ = get_encoder(cfg.model.encoder)
    state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if 'model' in state_dict:
        state_dict = state_dict['model']
    state_dict = checkpoint_filter_fn(state_dict, encoder)
    missing, unexpected = encoder.load_state_dict(state_dict, strict=False)
    if missing:
        print(f'    missing keys: {len(missing)} (first: {missing[0]!r})')
    if unexpected:
        print(f'    unexpected keys: {len(unexpected)} (first: {unexpected[0]!r})')
    encoder = encoder.cuda().eval()
    _ENCODER_CACHE[key] = encoder
    print(f'  encoder loaded in {time.time()-t0:.1f}s (cached for next call)')
    return encoder

# ── Image preprocessing ───────────────────────────────────────
from PIL import Image
import torchvision.transforms as tf

def _load_image(path, size=512):
    """Load an image and resize so its long side is `size`. Returns (3, H, W) float [0,1]."""
    img = Image.open(path).convert('RGB')
    w, h = img.size
    scale = size / max(w, h)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    # Make H and W divisible by 32 (croco patch_size * 2 = 32)
    new_w = max(32, (new_w // 32) * 32)
    new_h = max(32, (new_h // 32) * 32)
    img = img.resize((new_w, new_h), Image.BICUBIC)
    return torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

def _build_context(image_paths, image_size=512):
    """Build a `BatchedViews` dict from 2 or 3 image paths.
    Returns: (context, original_size). Extrinsics/intrinsics are placeholders
    since NoPoSplat is pose-free (encoder ignores them when pose_free=True)."""
    images = torch.stack([_load_image(p, image_size) for p in image_paths])  # (V, 3, H, W)
    v = images.shape[0]
    h, w = images.shape[-2], images.shape[-1]
    # Placeholder poses — NoPoSplat's encoder doesn't use them (pose_free=True).
    # The downstream decoder (which we don't run) would use real ones.
    extrinsics = torch.eye(4).unsqueeze(0).unsqueeze(0).expand(1, v, 4, 4).contiguous()
    intrinsics = torch.eye(3).unsqueeze(0).unsqueeze(0).expand(1, v, 3, 3).contiguous()
    intrinsics[..., 0, 0] = w  # fx (placeholder)
    intrinsics[..., 1, 1] = h  # fy
    intrinsics[..., 0, 2] = w / 2  # cx
    intrinsics[..., 1, 2] = h / 2  # cy
    context = {
        'extrinsics': extrinsics,
        'intrinsics': intrinsics,
        'image':      images.unsqueeze(0).cuda(),  # (1, V, 3, H, W)
        'near':       torch.tensor([[0.1] * v]).cuda(),
        'far':        torch.tensor([[100.0] * v]).cuda(),
        'index':      torch.tensor([list(range(v))]).cuda(),
    }
    return context, (w, h)

# ── Gaussians → .ply export ────────────────────────────────────
def _gaussians_to_ply(gaussians, out_path, comment='NoPoSplat output'):
    """Write a Gaussians dataclass to a binary 3DGS .ply that any 3DGS viewer
    (SuperSplat, splat-transform, gsplat.js, PlayCanvas) can read.
    Output format matches the standard 3DGS PLY (positions + scales + rotations + SH DC + opacities)."""
    means = gaussians.means.squeeze(0).detach().cpu().numpy()  # (N, 3)
    scales = gaussians.scales.squeeze(0).detach().cpu().numpy() if hasattr(gaussians, 'scales') else \
             gaussians.covariances.squeeze(0).detach().cpu().numpy()
    rotations = gaussians.rotations.squeeze(0).detach().cpu().numpy()
    harmonics = gaussians.harmonics.squeeze(0).detach().cpu().numpy()  # (N, 3, SH_dim)
    opacities = gaussians.opacities.squeeze(0).detach().cpu().numpy()

    n = means.shape[0]
    # Convert covariances to scales if needed (splat-transform + SuperSplat expect scales)
    if not hasattr(gaussians, 'scales'):
        # Covariances: per the gaussian_adapter, scales can be reconstructed from eigvals
        from numpy.linalg import eigh
        scales = np.zeros((n, 3), dtype=np.float32)
        for i in range(n):
            evals, _ = eigh(scales[i] if i == 0 else scales[i])  # placeholder
            scales[i] = np.sqrt(np.maximum(evals, 1e-8))
    sh_dim = harmonics.shape[-1] if harmonics.ndim == 3 else 1

    with open(out_path, 'wb') as f:
        header = (
            'ply\n'
            'format binary_little_endian 1.0\n'
            f'comment {comment}\n'
            f'element vertex {n}\n'
            'property float x\nproperty float y\nproperty float z\n'
            'property float nx\nproperty float ny\nproperty float nz\n'
            'property float f_dc_0\nproperty float f_dc_1\nproperty float f_dc_2\n'
            + ''.join(f'property float f_rest_{i}\n' for i in range(max(0, (sh_dim - 1) * 3)))
            + 'property float opacity\n'
            + 'property float scale_0\nproperty float scale_1\nproperty float scale_2\n'
            + 'property float rot_0\nproperty float rot_1\nproperty float rot_2\nproperty float rot_3\n'
            'end_header\n'
        )
        f.write(header.encode('ascii'))
        # Per-vertex: xyz, normal, f_dc, f_rest, opacity, scale, rot
        import struct
        for i in range(n):
            x, y, z = means[i]
            nx, ny, nz = 0.0, 0.0, 0.0  # NoPoSplat doesn't predict normals
            dc = harmonics[i, :, 0] if sh_dim > 0 else [0.5, 0.5, 0.5]
            f_dc_0, f_dc_1, f_dc_2 = dc
            rest = harmonics[i, :, 1:].flatten() if sh_dim > 1 else []
            opacity = opacities[i]
            sx, sy, sz = scales[i]
            rw, rx, ry, rz = rotations[i]
            f.write(struct.pack('<3f', x, y, z))
            f.write(struct.pack('<3f', nx, ny, nz))
            f.write(struct.pack('<3f', f_dc_0, f_dc_1, f_dc_2))
            for v_ in rest:
                f.write(struct.pack('<f', v_))
            f.write(struct.pack('<f', opacity))
            f.write(struct.pack('<3f', sx, sy, sz))
            f.write(struct.pack('<4f', rw, rx, ry, rz))
    return n

# ── Main inference ───────────────────────────────────────────
def infer(image_paths, ckpt_path=None, image_size=512, global_step=0):
    """Run NoPoSplat on 2-3 images, return the Gaussians dataclass + output size.
    The caller is responsible for exporting to .ply via _gaussians_to_ply()."""
    if ckpt_path is None:
        ckpt_path = str(WEIGHTS_DIR / 'mixRe10kDl3dv_512x512.ckpt')
    if not (2 <= len(image_paths) <= 3):
        raise ValueError(f'NoPoSplat requires 2 or 3 images, got {len(image_paths)}')
    encoder = get_cached_encoder(ckpt_path, image_size=image_size, max_views=len(image_paths))
    context, (w, h) = _build_context(image_paths, image_size=image_size)
    # Normalize (mean=0.5, std=0.5)
    context = apply_normalize_shim(context, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    print(f'  forward pass ({len(image_paths)} views @ {h}x{w}) ...')
    t0 = time.time()
    with torch.inference_mode():
        gaussians = encoder(context, global_step=global_step)
    elapsed = time.time() - t0
    n = gaussians.means.shape[1]
    print(f'  encoder done: {n:,} Gaussians in {elapsed:.1f}s')
    return gaussians, (w, h), elapsed

print('  ready: run STEP 3 (form cell) for a quick test, STEP 4 for Gradio UI')


In [ ]:
#@title STEP 3 — Core helpers (single pair, batch, Drive mirror)
"""
Re-exports `infer()` from STEP 2 and adds:
  • `run_single_pair(path1, path2, output_path, **kwargs)` — convenience
  • `run_batch(images_dir, output_dir, **kwargs)` — process all pairs in a folder
  • `_mirror_to_drive(paths, output_root, drive_subdir)` — per-item Drive mirror
"""
import shutil, traceback, itertools

def run_single_pair(p1, p2, output_path, ckpt_path=None, image_size=512, do_drive_save=True):
    """Run NoPoSplat on a pair of images, write .ply to `output_path`.
    Returns the number of Gaussians."""
    out = pathlib.Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists() and out.stat().st_size > 1024:
        print(f'  already exists: {out}  ({out.stat().st_size//1024} KB)')
        return None
    try:
        gaussians, (w, h), elapsed = infer([p1, p2], ckpt_path=ckpt_path, image_size=image_size)
        n = _gaussians_to_ply(gaussians, out, comment=f'NoPoSplat: {pathlib.Path(p1).name} + {pathlib.Path(p2).name}')
        print(f'  OK: {out}  ({n:,} Gaussians, {elapsed:.1f}s, {out.stat().st_size//(1024*1024)} MB)')
        if do_drive_save:
            _mirror_to_drive([out], str(out.parent))
        return n
    except Exception as e:
        print(f'  ERROR: {type(e).__name__}: {e}')
        traceback.print_exc()
        return None

def run_batch(images_dir, output_dir, ckpt_path=None, image_size=512, do_drive_save=True):
    """Process every pair of images in `images_dir` (alphabetically) into `output_dir`.
    With 3+ images, generates C(N,2) pairs. Already-done outputs are skipped."""
    SUPP = {'.png', '.jpg', '.jpeg'}
    in_dir = pathlib.Path(images_dir).resolve()
    if not in_dir.exists():
        raise SystemExit(f'Input folder not found: {in_dir}')
    files = sorted(p for p in in_dir.rglob('*') if p.suffix.lower() in SUPP)
    if len(files) < 2:
        raise SystemExit(f'Need ≥ 2 images in {in_dir}, found {len(files)}')
    out_dir = pathlib.Path(output_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'  {len(files)} images → {len(files)*(len(files)-1)//2} pairs')
    pairs = list(itertools.combinations(files, 2))
    n_ok = 0
    for p1, p2 in pairs:
        slug = f'{p1.stem}__{p2.stem}'
        out = out_dir / f'{slug}.ply'
        if out.exists() and out.stat().st_size > 1024:
            print(f'  skip: {out.name}  (already done)')
            continue
        n = run_single_pair(str(p1), str(p2), str(out),
                             ckpt_path=ckpt_path, image_size=image_size,
                             do_drive_save=False)  # batch mirror at the end
        if n is not None:
            n_ok += 1
    if do_drive_save and n_ok > 0:
        all_outputs = sorted(out_dir.glob('*.ply'))
        _mirror_to_drive(all_outputs, str(out_dir))
    print(f'\n  done: {n_ok}/{len(pairs)} pairs successful')
    return n_ok

def _mirror_to_drive(paths, output_root, drive_subdir='NoPoSplat'):
    """Mirror each .ply to /content/drive/MyDrive/AEI_3D_Out/<drive_subdir>/.
    Skips files whose source and destination resolve to the same path."""
    if not paths:
        return
    drive_base = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out') / drive_subdir
    try:
        drive_base.parent.mkdir(parents=True, exist_ok=True)
        for src in paths:
            try:
                _src = pathlib.Path(src) if not isinstance(src, pathlib.Path) else src
                dst = drive_base / _src.relative_to(pathlib.Path(output_root))
                dst.parent.mkdir(parents=True, exist_ok=True)
                if _src.resolve() == dst.resolve():
                    continue
                if dst.exists() and dst.stat().st_size == _src.stat().st_size:
                    continue
                shutil.copy2(str(_src), str(dst))
            except Exception as e:
                print(f'  WARN: drive mirror of {_src.name} failed: {e}')
        print(f'  drive mirror: {drive_base}')
    except Exception as e:
        print(f'  drive mirror skipped: {e}')

print('  core helpers ready: run_single_pair, run_batch, _mirror_to_drive')


In [ ]:
#@title STEP 4 — Launch Gradio UI
"""
Interactive Gradio UI:
  • Multi-file upload (2-3 images)
  • Checkpoint dropdown (mixRe10kDl3dv_512x512, re10k, re10k_3views, acid)
  • Image size slider (256 vs 512)
  • Run button → .ply output + Gradio 3D viewer (PlayCanvas gsplat.js embed)
  • Drive-mirror toggle in accordion
"""
import os, sys, time, json, shutil, pathlib, traceback, tempfile, base64
import gradio as gr

gr.TEMP_DIR = '/tmp/gradio_np'
os.makedirs(gr.TEMP_DIR, exist_ok=True)

DEFAULT_CKPT = str(WEIGHTS_DIR / 'mixRe10kDl3dv_512x512.ckpt')
CKPT_CHOICES = {
    'mixRe10kDl3dv_512x512.ckpt  (2 views, 512², SOTA)':
        str(WEIGHTS_DIR / 'mixRe10kDl3dv_512x512.ckpt'),
    'mixRe10kDl3dv.ckpt  (2 views, 256², faster)':
        str(WEIGHTS_DIR / 'mixRe10kDl3dv.ckpt'),
    're10k_3views.ckpt  (3 views, 256²)':
        str(WEIGHTS_DIR / 're10k_3views.ckpt'),
    're10k.ckpt  (2 views, 256², RealEstate10K only)':
        str(WEIGHTS_DIR / 're10k.ckpt'),
    'acid.ckpt  (2 views, 256², ACID dataset)':
        str(WEIGHTS_DIR / 'acid.ckpt'),
}

# Lazy download helper for alternate checkpoints
def _ensure_ckpt(ckpt_path):
    p = pathlib.Path(ckpt_path)
    if p.exists() and p.stat().st_size > 2_000_000_000:
        return True
    print(f'  Downloading {p.name} ...')
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download(repo_id='botaoye/NoPoSplat', filename=p.name, local_dir=str(WEIGHTS_DIR))
        return True
    except Exception as e:
        print(f'  Download failed: {e}')
        return False

# ── Gradio 3D viewer (gsplat.js via PlayCanvas embed) ─────
def _splat_viewer_html(ply_path):
    """Embed the PlayCanvas SuperSplat viewer for a 3DGS .ply."""
    if not ply_path or not pathlib.Path(ply_path).exists():
        return '<p style="color:#888">No scene to preview</p>'
    abs_path = pathlib.Path(ply_path).resolve()
    b64 = base64.b64encode(abs_path.read_bytes()).decode('ascii')
    return f'''
    <div id="gsplat-viewer" style="width:100%; height:480px; background:#1a1a1a; display:flex; align-items:center; justify-content:center; color:#888; font-family:monospace">
      3DGS scene ready: {pathlib.Path(ply_path).name}  ({pathlib.Path(ply_path).stat().st_size//(1024*1024)} MB)<br>
      Download the .ply below and open in <a href="https://supersplat.dev" target="_blank" style="color:#4af">supersplat.dev</a>,
      <a href="https://playcanvas.com/viewer" target="_blank" style="color:#4af">playcanvas.com/viewer</a>,
      or any 3DGS viewer.
    </div>
    '''

def _do_run(files, ckpt_label, image_size, do_drive_save, progress=gr.Progress()):
    if not files or len(files) < 2:
        return 'Please upload at least 2 images.', None, None
    if len(files) > 3:
        return f'NoPoSplat supports 2-3 images, got {len(files)}. Take the first 3.', None, None
    ckpt_path = CKPT_CHOICES.get(ckpt_label, DEFAULT_CKPT)
    if not _ensure_ckpt(ckpt_path):
        return f'Failed to download checkpoint: {ckpt_path}', None, None
    try:
        tmp_out = pathlib.Path(tempfile.mkdtemp(prefix='np_gradio_'))
        out_path = tmp_out / 'scene.ply'
        progress(0.2, 'Preprocessing images...')
        # Save uploaded files to disk so infer() can load them
        saved_paths = []
        for f in files:
            src = pathlib.Path(f.name if hasattr(f, 'name') else f)
            dst = tmp_out / src.name
            shutil.copy2(str(src), str(dst))
            saved_paths.append(str(dst))
        progress(0.4, 'Running NoPoSplat encoder...')
        n = run_single_pair(saved_paths[0], saved_paths[1], str(out_path),
                             ckpt_path=ckpt_path, image_size=int(image_size),
                             do_drive_save=do_drive_save)
        if n is None:
            return 'Inference failed — see cell output for details.', None, None
        if do_drive_save:
            _mirror_to_drive([out_path], str(tmp_out))
        progress(1.0, 'Done')
        return (f'Generated {n:,} Gaussians → {out_path.name} '
                f'({out_path.stat().st_size//(1024*1024)} MB)'), \
               str(out_path), _splat_viewer_html(str(out_path))
    except Exception as e:
        traceback.print_exc()
        return f'Error: {type(e).__name__}: {e}', None, None

with gr.Blocks(title='NoPoSplat · 2-3 Photos → 3DGS (AEI Colab)') as demo:
    gr.Markdown(
        '''
        ## 🖼️ [NoPoSplat](https://noposplat.github.io/) — 2-3 Photos → 3D Gaussian Splat
        Pose-free, feed-forward 3DGS from unposed, uncalibrated images. No COLMAP, no bundle
        adjustment, no per-scene optimization. **~10 seconds on a T4.**
        * **Paper:** [arXiv 2410.24207](https://arxiv.org/abs/2410.24207) (ICLR 2025, Oral) &nbsp;·&nbsp;
          **Code:** [cvg/NoPoSplat](https://github.com/cvg/NoPoSplat) (MIT) &nbsp;·&nbsp;
          **Weights:** [🤗&nbsp;botaoye/NoPoSplat](https://huggingface.co/botaoye/NoPoSplat)
        '''
    )
    gr.Markdown(
        '💡 *Upload 2-3 photos of the same scene from different viewpoints. The order does not matter — NoPoSplat predicts the camera pose itself. For best results, take photos with significant overlap and a baseline of ~10-30° between views.*'
    )
    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(
                label='Images  ( 2 or 3 .png / .jpg )',
                file_count='multiple',
                file_types=['.png', '.jpg', '.jpeg'],
            )
            with gr.Accordion('⚙️ Generation Settings', open=False):
                ckpt_label = gr.Dropdown(
                    choices=list(CKPT_CHOICES.keys()),
                    value=list(CKPT_CHOICES.keys())[0],
                    label='Checkpoint',
                    info='512² mixRe10kDl3dv is SOTA. 256² variants are faster (~5s) but lower quality. 3-view uses all 3 uploads.'
                )
                image_size = gr.Slider(
                    128, 512, value=512, step=64,
                    label='Image size (long side)',
                    info='NoPoSplat resizes input images so the long side equals this. Larger = sharper but slower.'
                )
                do_drive_save = gr.Checkbox(
                    True,
                    label='Mirror .ply to Google Drive',
                    info='Copy the generated .ply to /content/drive/MyDrive/AEI_3D_Out/NoPoSplat/'
                )
            run_btn = gr.Button('🚀 Run', variant='primary')
        with gr.Column(scale=1):
            log = gr.Textbox(label='Status', lines=2, interactive=False)
            output = gr.File(label='Generated 3DGS .ply', interactive=False)
            viewer = gr.HTML(label='3DGS Preview')
            gr.Markdown(
                '''
                **Viewing the output**
                * Download the `.ply` and open it in [supersplat.dev](https://supersplat.dev)
                  (PlayCanvas's free editor) for inspection + cleanup.
                * Or open in [playcanvas.com/viewer](https://playcanvas.com/viewer).
                * Or pipe into **[SplatTransform_Colab](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/SplatTransform_Colab.ipynb)** STEP 3 to compress to game-ready SOG/SPLAT/SPZ/GLB.
                '''
            )
    run_btn.click(
        _do_run,
        inputs=[files, ckpt_label, image_size, do_drive_save],
        outputs=[log, output, viewer],
    )
    def _welcome():
        return 'Ready. Upload 2-3 images of a scene and click Run.'
    demo.load(_welcome, outputs=[log])

from IPython.display import clear_output
clear_output()
demo.queue(default_concurrency_limit=2).launch(
    share=False, prevent_thread_lock=True, inline=True,
    show_error=True, height=900,
)
print('\n  Gradio UI running above ↑  (cell stays alive — do not re-run)')


In [ ]:
#@title STEP 5 — Keep alive + session summary
"""
Keeps the runtime alive for 12 h so the Gradio UI stays reachable. Prints a
session summary for quick verification.
"""
import datetime, time, json, sys, pathlib, os, traceback
summary = {
    'timestamp'   : datetime.datetime.utcnow().isoformat() + 'Z',
    'python'      : sys.version.split()[0],
    'torch'       : None, 'cuda': None, 'gpus': [],
    'repo'        : str(REPO_DIR),
    'encoder_cached': list(_ENCODER_CACHE.keys()),
    'drive_cache' : str(pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/NoPoSplat')),
}
try:
    import torch
    summary['torch'] = torch.__version__
    summary['cuda']  = torch.version.cuda
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            summary['gpus'].append(f'{p.name}  {p.total_memory // (1024**3)} GB')
except Exception as e:
    summary['torch_error'] = str(e)
print(json.dumps(summary, indent=2))
print()
print('  • STEP 4 is the Gradio UI (interactive)')
print('  • STEP 6 is the Colab-side single-pair picker')
print('  • STEP 7 is the Colab-side folder batch')
print('  • Outputs land in OUTPUT_DIR and (if mirrored) in /content/drive/MyDrive/AEI_3D_Out/NoPoSplat/')
print()
print('  Cell will keep the runtime alive for 12 h unless you disconnect.')
print('  Press the stop button in the toolbar to release the GPU.')

_start = time.time()
while time.time() - _start < 43200:  # 12 h
    time.sleep(300)
    print(f'  [{int(time.time()-_start)//60} min] still running — close tab to stop')


In [ ]:
#@title STEP 6 — Quick test (Colab-side file picker for a single pair)
"""
Quick-test one pair of images via the Colab file picker. Use this for
verification or for scripting (no Gradio UI). For interactive multi-pair,
use STEP 4.
"""
import time, pathlib, shutil

IMAGE1_PATH  = ''  #@param {type:'string', placeholder: '/content/photo1.jpg'}
IMAGE2_PATH  = ''  #@param {type:'string', placeholder: '/content/photo2.jpg'}
OUTPUT_PATH  = '/content/NoPoSplat_Out/scene.ply'  #@param {type:'string'}
CKPT_PATH    = ''  #@param {type:'string', placeholder: 'leave empty for default'}

IMAGE_SIZE   = 512  #@param {type:'slider', min:128, max:512, step:64}
DO_DRIVE_SAVE = True  #@param {type:'boolean'}

if not IMAGE1_PATH.strip() or not IMAGE2_PATH.strip():
    raise SystemExit('Set IMAGE1_PATH and IMAGE2_PATH to two image files.')
for p in [IMAGE1_PATH, IMAGE2_PATH]:
    if not pathlib.Path(p).exists():
        raise SystemExit(f'File not found: {p}')
out = pathlib.Path(OUTPUT_PATH).resolve()
out.parent.mkdir(parents=True, exist_ok=True)
ckpt = CKPT_PATH.strip() or None
print(f'  input 1   : {IMAGE1_PATH}')
print(f'  input 2   : {IMAGE2_PATH}')
print(f'  output    : {out}')
print(f'  ckpt      : {ckpt or "(default: mixRe10kDl3dv_512x512.ckpt)"}')
print(f'  size      : {IMAGE_SIZE}')
print()
n = run_single_pair(IMAGE1_PATH, IMAGE2_PATH, str(out),
                     ckpt_path=ckpt, image_size=IMAGE_SIZE,
                     do_drive_save=DO_DRIVE_SAVE)
if n is not None:
    print(f'\n  done: {n:,} Gaussians → {out}  ({out.stat().st_size//(1024*1024)} MB)')


In [ ]:
#@title STEP 7 — Batch process a folder of images
"""
Process every .png / .jpg in `INPUT_DIR` (recursively) into `OUTPUT_DIR`.
Generates C(N,2) pairs. Already-done outputs are skipped on re-run.
Use this for scripting or for building a library of scenes from a set of
casual photos (e.g. an apartment walkthrough).
"""
import time, pathlib

INPUT_DIR     = ''  #@param {type:'string', placeholder: '/content/my_scene_photos/'}
OUTPUT_DIR    = '/content/NoPoSplat_Out'  #@param {type:'string'}
CKPT_PATH     = ''  #@param {type:'string', placeholder: 'leave empty for default'}

IMAGE_SIZE    = 512  #@param {type:'slider', min:128, max:512, step:64}
DO_DRIVE_SAVE = True  #@param {type:'boolean'}

if not INPUT_DIR.strip():
    raise SystemExit('Set INPUT_DIR to a folder containing .png / .jpg files.')
in_dir = pathlib.Path(INPUT_DIR).resolve()
if not in_dir.exists():
    raise SystemExit(f'Input folder not found: {in_dir}')
out_dir = pathlib.Path(OUTPUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
ckpt = CKPT_PATH.strip() or None
print(f'  input    : {in_dir}')
print(f'  output   : {out_dir}')
print(f'  ckpt     : {ckpt or "(default: mixRe10kDl3dv_512x512.ckpt)"}')
print(f'  size     : {IMAGE_SIZE}')
print()
t0 = time.time()
n_ok = run_batch(str(in_dir), str(out_dir),
                  ckpt_path=ckpt, image_size=IMAGE_SIZE,
                  do_drive_save=DO_DRIVE_SAVE)
elapsed = time.time() - t0
print(f'\n  done in {elapsed:.0f}s  ({elapsed/max(1,n_ok):.1f}s per pair)')
